# MPECSS - NosBench Benchmark (Group 3 of 3)

This notebook runs **Group 3** of the NosBench benchmark suite.

- **Dataset**: NosBench (Group 3)
- **Problems**: ~201 (balanced small/medium/large split)
- **Resume support**: built in via `kaggle_setup/resumable_benchmark.py`

**Prerequisites:**
1. Add the `mpecss-benchmarks` dataset to this notebook
2. Enable Internet access

Run the cells in order. After a Kaggle restart, re-run cells 1-3, then use the **Resume** cell.

## 1. Configuration

In [ ]:
# NosBench Group 3 Configuration
import os

DATASET = 'nosbench'
GROUP = 3
RUN_TAG = f'Kaggle_NosBench_Group{GROUP}'

WORKERS = 4
TIMEOUT = 3600
SAVE_LOGS = True
SHUFFLE = True

# Repository (code)
REPO_DIR = "/kaggle/working/MPECSSCODEPAPER"
REPO_GIT_URL = "https://github.com/mrsaurabhtanwar/MPECSSCODEPAPER.git"

# IMPORTANT: Results go directly to /kaggle/working/outputs to persist after session ends
OUTPUT_DIR = "/kaggle/working/outputs"

# Benchmark data path (from Kaggle dataset - NOT from repo)
DATASET_PATH = '/kaggle/input/datasets/mrsaurabhtanwar/mpecss-benchmarks/benchmarks/nosbench/nosbench-json'

# Problem list for this group (from repo)
PROBLEM_LIST = f"{REPO_DIR}/kaggle_setup/nosbench_splits/nosbench_group{GROUP}_problems.txt"

# Verify dataset is attached
if not os.path.exists(DATASET_PATH):
    print("[ERROR] Dataset not found!")
    print("Please add the 'mpecss-benchmarks' dataset to this notebook.")
else:
    print(f"[OK] Dataset found: {DATASET_PATH}")
    print(f"[OK] Results will save to: {OUTPUT_DIR}")

## 2. Prepare The Repository

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

repo_path = Path(REPO_DIR)
REPO_COMMIT = "SET_RELEASE_COMMIT"
if REPO_COMMIT == "SET_RELEASE_COMMIT":
    raise RuntimeError("Set REPO_COMMIT to the pushed release commit before the final Kaggle run.")

if repo_path.exists():
    shutil.rmtree(repo_path)

print(f"Cloning: {REPO_GIT_URL} @ {REPO_COMMIT}")
subprocess.run(["git", "clone", "--depth", "1", REPO_GIT_URL, str(repo_path)], check=True)
subprocess.run(["git", "checkout", REPO_COMMIT], check=True, cwd=str(repo_path))
sys.path.insert(0, str(repo_path))
print(f"[OK] Repository ready")

## 3. Install Dependencies

In [ ]:
%%bash
cd /kaggle/working/MPECSSCODEPAPER
pip install -q --upgrade pip
pip install -q -e .
echo "[OK] Installation complete"

## 4. Data Path Setup

In [ ]:
import os

print(f"Dataset path: {DATASET_PATH}")
print(f"Problem list: {PROBLEM_LIST}")

if os.path.exists(DATASET_PATH):
    count = len([f for f in os.listdir(DATASET_PATH) if f.endswith('.json')])
    print(f"[OK] Total problems in dataset: {count}")
else:
    print(f"[ERROR] Dataset path not found!")

if os.path.exists(PROBLEM_LIST):
    with open(PROBLEM_LIST) as f:
        group_count = sum(1 for line in f if line.strip() and not line.startswith('#'))
    print(f"[OK] Problems in Group {GROUP}: {group_count}")
else:
    print(f"[ERROR] Problem list not found!")

## 5. Inspect The Kaggle Instance

In [ ]:
import multiprocessing
import platform
import psutil

print("=" * 60)
print("System Information")
print("=" * 60)
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"Logical CPU cores: {multiprocessing.cpu_count()}")

mem = psutil.virtual_memory()
print(f"Total RAM: {mem.total / 1024**3:.1f} GB")
print(f"Available RAM: {mem.available / 1024**3:.1f} GB")
print("=" * 60)

## 6. Run Preflight Checks

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/MPECSSCODEPAPER
python scripts/preflight_checks.py

## 7. Load The Runner Helper

In [ ]:
import subprocess
import sys

def run_benchmark(resume=False, summary=False):
    cmd = [
        sys.executable,
        f"{str(REPO_DIR)}/kaggle_setup/resumable_benchmark.py",
        "--dataset", DATASET,
        "--repo-dir", str(REPO_DIR),
        "--tag", RUN_TAG,
        "--workers", str(WORKERS),
        "--timeout", str(TIMEOUT),
        "--path", str(DATASET_PATH),
        "--problem-list", str(PROBLEM_LIST),
        "--output-dir", str(OUTPUT_DIR),  # CRITICAL: Save to persistent location
    ]
    if SAVE_LOGS: cmd.append("--save-logs")
    if SHUFFLE: cmd.append("--shuffle")
    if resume: cmd.append("--resume-latest")
    if summary: cmd.append("--summary-only")
    
    print("+ " + " ".join(str(x) for x in cmd))
    subprocess.run(cmd)

## 8. Launch A Fresh Run

In [ ]:
run_benchmark()

## 9. Resume After A Kaggle Restart

In [ ]:
run_benchmark(resume=True)

## 10. Progress Summary

In [ ]:
run_benchmark(summary=True)

## 11. Copy Results For Download

In [ ]:
%%bash
set -euo pipefail
mkdir -p /kaggle/working/outputs
cp -r /kaggle/working/MPECSSCODEPAPER/results/* /kaggle/working/outputs/ || true

echo "[OK] Results copied to /kaggle/working/outputs/"
echo "Download from the Kaggle file browser or output panel"

## 12. Software Versions For Your Paper

In [ ]:
import casadi
import matplotlib
import numpy
import pandas
import platform
import psutil
import scipy

print("Software Versions")
print("=" * 40)
print(f"Python: {platform.python_version()}")
print(f"NumPy: {numpy.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Pandas: {pandas.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"CasADi: {casadi.__version__}")
print(f"psutil: {psutil.__version__}")
print("=" * 40)